# Data Cleaning Notebook: fake_job_postings.csv

This notebook was automatically generated with custom cleaning steps.

**Instructions:**
1. Execute each cell in order (Shift+Enter)
2. Review the output and modify code as needed
3. The final cell will export your cleaned data

**Original file:** `fake_job_postings.csv`

**Selected cleaning steps:** imports, load_csv, preview, missingness, normalize_text, split_location, parse_salary, normalize_binary, drop_duplicates, save_cleaned


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


In [ ]:
# Load the CSV file
csv_path = r"D:\Gannon\4semester\jupyter\notebook-studio\app\ipynb\uploads\1777313505416_fake_job_postings.csv"
df = pd.read_csv(csv_path)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")


In [ ]:
# Display basic information about the dataset
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nDataset shape:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\nColumn names:")
print(df.columns.tolist())

print("\nColumn data types:")
print(df.dtypes)

print("\nBasic statistics:")
display(df.describe())


In [ ]:
# Check for missing values
print("=" * 50)
print("MISSING VALUES ANALYSIS")
print("=" * 50)

missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Percentage': missing_percent
})

missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print(f"\nColumns with missing values:")
    display(missing_df)
    
    # Fill numeric columns with mean
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].mean(), inplace=True)
            print(f"Filled missing values in '{col}' with mean")
    
    # Fill categorical columns with mode
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown', inplace=True)
            print(f"Filled missing values in '{col}' with mode")
else:
    print("\nNo missing values found!")

print(f"\nMissing values after cleaning: {df.isnull().sum().sum()}")


In [ ]:
# Normalize text columns
print("=" * 50)
print("TEXT NORMALIZATION")
print("=" * 50)

text_cols = df.select_dtypes(include=['object']).columns

for col in text_cols:
    # Strip whitespace
    df[col] = df[col].str.strip()
    # Convert to lowercase (optional - comment out if not needed)
    # df[col] = df[col].str.lower()
    print(f"Normalized text in column: {col}")

print(f"\nText normalization complete for {len(text_cols)} columns")


In [ ]:
# Split location column into components
print("=" * 50)
print("LOCATION SPLITTING")
print("=" * 50)

# Check if location column exists
location_col = None
for col in df.columns:
    if 'location' in col.lower():
        location_col = col
        break

if location_col:
    print(f"Found location column: {location_col}")
    
    # Split by comma
    location_split = df[location_col].str.split(',', expand=True)
    
    if location_split.shape[1] >= 1:
        df['city'] = location_split[0].str.strip()
        print("Created 'city' column")
    
    if location_split.shape[1] >= 2:
        df['state'] = location_split[1].str.strip()
        print("Created 'state' column")
    
    if location_split.shape[1] >= 3:
        df['country'] = location_split[2].str.strip()
        print("Created 'country' column")
    
    print(f"\nLocation split complete!")
    display(df[['city', 'state'] + (['country'] if 'country' in df.columns else [])].head())
else:
    print("No location column found - skipping split")


In [ ]:
# Parse salary range fields
print("=" * 50)
print("SALARY PARSING")
print("=" * 50)

# Find salary-related columns
salary_cols = [col for col in df.columns if 'salary' in col.lower()]

if salary_cols:
    print(f"Found salary columns: {salary_cols}")
    
    for col in salary_cols:
        # Extract numeric values from salary strings
        df[f'{col}_numeric'] = df[col].astype(str).str.extract(r'(\d+)')[0]
        df[f'{col}_numeric'] = pd.to_numeric(df[f'{col}_numeric'], errors='coerce')
        print(f"Created numeric column: {col}_numeric")
    
    print(f"\nSalary parsing complete!")
    display(df[[col for col in df.columns if 'salary' in col.lower()]].head())
else:
    print("No salary columns found - skipping parsing")


In [ ]:
# Normalize binary columns
print("=" * 50)
print("BINARY NORMALIZATION")
print("=" * 50)

# Find binary columns (Yes/No, True/False, etc.)
for col in df.select_dtypes(include=['object']).columns:
    unique_vals = df[col].dropna().unique()
    
    # Check if column has only 2 unique values
    if len(unique_vals) == 2:
        # Map to 1/0
        val1, val2 = unique_vals[0], unique_vals[1]
        
        # Determine which value should be 1
        if str(val1).lower() in ['yes', 'true', 't', '1']:
            df[col] = df[col].map({val1: 1, val2: 0})
        elif str(val2).lower() in ['yes', 'true', 't', '1']:
            df[col] = df[col].map({val1: 0, val2: 1})
        else:
            df[col] = df[col].map({val1: 1, val2: 0})
        
        print(f"Normalized binary column '{col}': {val1}/{val2} → 1/0")

print("\nBinary normalization complete!")


In [ ]:
# Remove duplicate rows
print("=" * 50)
print("DUPLICATE DETECTION AND REMOVAL")
print("=" * 50)

initial_rows = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
final_rows = len(df)
duplicates_removed = initial_rows - final_rows

print(f"\nInitial rows: {initial_rows}")
print(f"Final rows: {final_rows}")
print(f"Duplicates removed: {duplicates_removed}")

if duplicates_removed > 0:
    print(f"Removed {duplicates_removed} duplicate rows ({(duplicates_removed/initial_rows)*100:.2f}%)")
else:
    print("No duplicates found!")


In [ ]:
# Export cleaned data
print("=" * 50)
print("EXPORT CLEANED DATA")
print("=" * 50)

# Generate output path
import os
base_name = os.path.splitext(os.path.basename(csv_path))[0]
output_dir = os.path.join(os.path.dirname(csv_path), '..', 'cleaned')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"{base_name}_cleaned.csv")

# Export to CSV
df.to_csv(output_path, index=False)

print(f"\nCleaned data exported successfully!")
print(f"Output file: {output_path}")
print(f"\nFinal dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\nColumn summary:")
display(pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes.values,
    'Non-Null': df.count().values,
    'Unique': [df[col].nunique() for col in df.columns]
}))
